In [ ]:
import sys
sys.path.append("../../legal-data-clustering/")
# %run '../../legal-data-clustering/pipeline/cd_preprocessing.py'
# %run '../../legal-data-clustering/pipeline/cd_cluster.py'
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
%run 'common.py'

import matplotlib.patches as patches
import numpy as np
import regex
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
plt.ioff()

In [ ]:
from matplotlib.colors import ListedColormap
import seaborn as sns

def cluster_family_colors(dataset):
    return sns.color_palette("tab20")

def cluster_family_plt_colors(dataset):
    colors = np.array(
            [np.array([*c, 1]) for c in cluster_family_colors(dataset)] +
            [np.array([0.75, 0.75, 0.75, 1])]
        )
    return ListedColormap(colors)

# Plot Functions

In [ ]:
def get_level_node_weights(G, level):
    '''
    Gets all weights of nodes of a given level.
    Used e.g. to obtain the total weight of level.
    '''
    return [data['weight'] for node_key, data in G.nodes(data=True) if data['bipartite'] == level]

def calc_config(G, spline_node_ratio=0.5):
    '''
    Calculate basic parameters to plot based on the input graph G.
    '''
    config = dict()
    config['levels'] = sorted(set(nx.get_node_attributes(G, 'bipartite').values()), key=lambda x: x.lower())
    config['levels_node_weight_sum'] = {
        level: 
        sum(get_level_node_weights(G, level))
        for level in config['levels']
    }
    config['x_weight_scale'] = max(config['levels_node_weight_sum'].values()) # number so chars that are represented by 100% of the width of the plot
    config['node_height'] = 1/len(config['levels'])*(1-spline_node_ratio) # height of nodes
    config['spline_height'] = 1/(len(config['levels'])-1)*spline_node_ratio # height of splines
    
    config['level_tick_positions'] = list(reversed([
        (config['node_height'] + config['spline_height']) * idx + config['node_height'] / 2
        for idx, level in enumerate(config['levels'])
    ]))
    return config

def calc_node_positions(G, config, level_node_orders):
    '''
    Calculates the positions of the nodes
    '''
    positions = {}

    for level_idx, level in enumerate(config['levels']):
        left = 0

        # Center
        left = (1 - config['levels_node_weight_sum'][level]/config['x_weight_scale']) / 2

        for node_idx, node in enumerate(level_node_orders[level]):
            height = config['node_height']
            width = G.nodes[node]['weight']/config['x_weight_scale']

            top = 1 - (config['node_height'] + config['spline_height']) * level_idx
            bottom = top - height

            positions[node] = dict(
                height=height,
                width=width,
                top=top,
                bottom=bottom,
                left=left,
            )

            left += width

    
    return positions

def calc_edge_positions(G, node_positions, config, level_node_orders):
    '''
    Calculates the position of the edges.
    '''
    edge_positions = {}

    edge_start_left_offset = {node: 0 for node in node_positions.keys()}
    edge_end_left_offset = {node: 0 for node in node_positions.keys()}

    global_node_order = list(itertools.chain.from_iterable(
        [level_node_orders[level] for level in config['levels']]
    ))

    edges = sorted(
        [(u, v) for (u, v) in G.edges], 
        key=lambda x: (
            global_node_order.index(x[0]),
            global_node_order.index(x[1]),
        ),
    )
    for u, v in edges:
        start_offset = edge_start_left_offset[u]
        end_offset = edge_end_left_offset[v]
        
        y0 = node_positions[u]['bottom']
        x0 = node_positions[u]['left'] + start_offset
        yn = node_positions[v]['top']
        xn = node_positions[v]['left'] + end_offset
        width = G.edges[u, v]['weight'] / config['x_weight_scale']

        edge_start_left_offset[u] += width
        edge_end_left_offset[v] += width

        edge_positions[(u, v)] = dict(
            x0=x0,
            xn=xn,
            y0=y0,
            yn=yn,
            width=width,
        )
        
    return edge_positions


def draw_node(ax, left, bottom, top, width, height, label=None, color='k', v_position='center', label_rotation=0):
    '''
    Draws a node
    '''
    patch = patches.Rectangle(
            (
                left,
                bottom,
            ),
            width,
            height,
            color=color
         )
    patch.set_edgecolor('k')  
    patch.set_linewidth('0')
    ax.add_patch(patch)
    
def draw_label(ax, left, bottom, top, width, height, label=None, color='k', v_position='center', label_rotation=0, min_font_size=3):
    if v_position == 'bottom':
        hpos = bottom + height/3
    elif v_position == 'top':
        hpos = bottom + height/3*2
    else:
        hpos = bottom + height/2

    if label:
        if label_rotation:
            fontsize = 2 + 6 * width / 0.02
            fontsize = min(fontsize, 8)
            if min_font_size:
                fontsize = max(fontsize, min_font_size)
            xoffset = 0.0006
            yoffset = 0
        else:
            fontsize = 6
            xoffset = 0
            yoffset = -0.0006
        ax.annotate(
            str(label),
            (left + width/2 + xoffset, hpos + yoffset),
            color='k', 
            weight='bold', 
            ha='center', 
            va='center',
            fontsize=fontsize,
            rotation=label_rotation,
        )
        
        
def get_blueprint_vals(resolution):
    '''
    Creates a blueprint that can be transformed to print the splines.
    '''
    x = np.array([0, 0.15, 0.5, 0.85, 1])
    y = np.linspace(0, 1, len(x))
    z = np.polyfit(y, x, 4)
    f = np.poly1d(z)
    blueprint_y_vals = np.linspace(y[0], y[-1], resolution)
    blueprint_x_vals = f(blueprint_y_vals)
    return blueprint_x_vals, blueprint_y_vals


blueprint_x_vals, blueprint_y_vals = get_blueprint_vals(50)


def draw_edge(ax, x0, xn, y0, yn, width, 
             blueprint_x_vals=blueprint_x_vals, blueprint_y_vals=blueprint_y_vals, 
             color='k', alpha=0.5, hatch=None):
    '''
    Draws a node
    '''
    y_scale = yn - y0
    ty = blueprint_y_vals * y_scale + y0
    x_scale = xn - x0
    tx = blueprint_x_vals * x_scale + x0
    y_new = np.concatenate([ty, ty[::-1], ])
    x_new = np.concatenate([tx, tx[::-1] + width, ])
    result = np.array([x_new, y_new]).transpose()
    ax.add_patch(
        patches.Polygon(result, facecolor=color, alpha=alpha, lw=0, hatch=hatch, edgecolor='k')
    )

# Preprocessing functions

## Load graph

In [ ]:
def filter_edges(G, threshold, attr='weight'):
    edges_to_remove = [
        (u, v) 
        for u, v, data in G.edges(data=True) 
        if (
            data[attr] < G.nodes[u][attr] * threshold or
            data[attr] < G.nodes[v][attr] * threshold 
        )
    ]
    H = G.copy()
    H.remove_edges_from(edges_to_remove)
    return H

def load_graph(path, nodes_per_year, edge_threshold):
    # Load graph
    orig_G = nx.read_gpickle(path)
    G = orig_G.copy()
    
    # Selecte property for weights in plot
    nx.set_node_attributes(G, nx.get_node_attributes(G, 'tokens_n'), 'weight')
    nx.set_edge_attributes(G, nx.get_edge_attributes(G, 'tokens_n'), 'weight')
    
    # Remove insignificant edges
    G = filter_edges(G, threshold=edge_threshold)
    
    # Get nodes by snapshot
    years = sorted(set(nx.get_node_attributes(G, 'bipartite').values()))
    node_years = [
        sorted(
            [
                (n, data['weight']) 
                for n, data in G.nodes(data=True) 
                if data['bipartite'] == year
            ], 
            key=lambda tup: tup[-1], reverse=True
        ) 
        for year in years
    ]
    
    # Split nodes in big nodes (displayed) and small nodes (summarized in misc)
    big_nodes = [
        node[0]
        for nodes in node_years
        for node in nodes[:nodes_per_year]
    ]
    
    small_nodes = [
        node[0]
        for nodes in node_years
        for node in nodes[nodes_per_year:]
    ]
    
    # Build graph in which small nodes are summarized in misc
    H = nx.subgraph(G, big_nodes).copy()
    
    # Map small nods to corresponding misc node
    small_nodes_mapper = dict() 
    for year, nodes in zip(years, node_years):
        for node, data in nodes[nodes_per_year:]:
            small_nodes_mapper[node] = f'misc_{year}'
    
    # calculate weight of misc nodes
    misc_weights = {f'misc_{year}': 0 for year in years}
    for small_node in small_nodes:
        weight = G.nodes[small_node]['weight']
        misc_node_key = small_nodes_mapper[small_node]
        misc_weights[misc_node_key] += weight
    
    # add misc nodes to graph
    H.add_nodes_from([
        (k, dict(weight=w, bipartite=k.split('_')[-1])) 
        for k, w in misc_weights.items()
    ])
       
    # get edges regarding misc nodes 
    edges_to_merge = [
        (u, v, d)
        for u, v, d in G.edges(data=True)
        if not (u in big_nodes and v in big_nodes)
    ]
        
    # calculate weight of misc edges
    for u, v, d in edges_to_merge:
        u_mapped = small_nodes_mapper.get(u, u)
        v_mapped = small_nodes_mapper.get(v, v)
        if H.has_edge(u_mapped, v_mapped):
            H.edges[u_mapped, v_mapped]['weight'] += d['weight']
        else:
            H.add_edge(u_mapped, v_mapped, weight=d['weight'])   
     
    # filter edges by absolute weight
# Outdated. We filter now by relativ weight to the source and target node. See above.
#     edges = [(u, v) for u, v, data in H.edges(data=True) if data['weight'] < edge_threshold]
#     print(f'Removes {len(edges)} of {len(H.edges)} edges ({len(edges)/len(H.edges)*100:.1f}%).')
#     H.remove_edges_from(edges)
#     print(f'Remaining {len(H.edges)/(len(years)-1)} edges per year mapping')
        
    return H, orig_G

## Order

In [ ]:
def order_nodes_by_weight(config, G):
    return {
        level:
        sorted([n for n, data in G.nodes(data=True) if data['bipartite'] == level], key=lambda n: (
               G.nodes[n]['weight'] 
               if not n.startswith('misc_')
               else -1
           ), reverse=True
        )  # Order by weight
        for level in config['levels']
    }

In [ ]:
def order_edges_by_weight(edge_positions):
    return sorted(
        edge_positions.items(), 
        key=lambda tup:(
            -1
            if tup[0][0].startswith('misc_') or tup[0][1].startswith('misc_')
            else tup[-1]['width']
        )
    )

## Color and label position

In [ ]:
def alternating_colors(edge_positions, node_positions, G, level_node_orders):
    '''
    sets colors in edge_positions
    '''
    # Alternating colors
    for nodes_at_level in level_node_orders.values():
        for idx, node in enumerate(nodes_at_level):
            if idx % 2 == 0:
                node_positions[node]['color'] = '0.7'
                node_positions[node]['v_position'] = 'top'
            else:
                node_positions[node]['color'] = '0.6'
                node_positions[node]['v_position'] = 'bottom'

            for out_edge in G.out_edges(node):
                    edge_positions[out_edge]['color'] = node_positions[node]['color']
                    
    # Special color for misc nodes and corresponding edges
    for node in node_positions:
        if node.startswith('misc_'):
            color ='0.8'
            node_positions[node]['color'] = color
            node_positions[node]['v_position'] = 'center'
            for out_edge in [
                *G.out_edges(node),
                *G.in_edges(node)
            ]:
                    edge_positions[out_edge]['color'] = color

In [ ]:
def color_merges_splits(edge_positions, node_positions, H,):
    '''
    Coloring merges, splits, etc.
    Colors:
    - merge edges: green
    - split edges: red
    - combined merge and split edges: yellow
    - ignored edges: blue
    A edge is considered a merge/split edge if the source/target node has a degree > 1 of significant edges. 
    An edge is significant if its weight is a least the node weight * threshold of the source and target node.
    Edges that are not significant are ignored edges.
    Edges from or to Misc. nodes are not recolored.
    '''
    for edge_key in edge_positions:
        u, v = edge_key
        if u.startswith('misc_') and v.startswith('misc_'):
            continue
        if H.has_edge(u, v):
            is_u_multi = H.out_degree(u) > 1 and not u.startswith('misc_')
            is_v_multi = H.in_degree(v) > 1 and not v.startswith('misc_')
            if is_u_multi and is_v_multi:
                edge_positions[edge_key]['color'] = 'magenta'
                edge_positions[edge_key]['hatch'] = 'X'
            elif is_u_multi:
                # Split
                edge_positions[edge_key]['color'] = 'red'
                edge_positions[edge_key]['hatch'] = '/'
            elif is_v_multi:
                # Merge
                edge_positions[edge_key]['color'] = 'blue'
                edge_positions[edge_key]['hatch'] = '\\'
        else:
            # This is not used if remove insifnificant edges in the first place 
            edge_positions[edge_key]['color'] = 'orange'
            edge_positions[edge_key]['hatch'] = '|'

In [ ]:
def color_births_deaths(node_positions, H):
    years = sorted(set(nx.get_node_attributes(H, 'bipartite').values()))
    for node in node_positions:
        is_birth = H.nodes[node]['bipartite'] != years[0] and H.in_degree(node) == 0
        is_death = H.nodes[node]['bipartite'] != years[-1] and H.out_degree(node) == 0
        if is_birth and is_death:
            pass
            # clusters living in one year only are not highlighted if statement below is commented out
            # node_positions[node]['color'] = 'orange'
        elif is_birth:
            node_positions[node]['color'] = 'gold'
        elif is_death:
            node_positions[node]['color'] = 'chocolate'

In [ ]:
def color_by_cluster_family(orig_G, node_positions, edge_positions, dataset, edge_threshold=.15):
    components = cluster_families(orig_G, threshold=edge_threshold)
    cmap = cluster_family_plt_colors(dataset)
    
    components = components[:20]
    
    order = [c[0] for c in components]
    
    for order_nr, nodes in enumerate(components):
        color = cmap(order_nr)
        for node in nodes:
            if node in node_positions:
                node_positions[node]['color'] = color

        for edge in edge_positions:
            u, v = edge
            if u in nodes or v in nodes:
                 edge_positions[edge]['color'] = color

## Run

In [ ]:
def plot_evolution_graph(dataset, config_str, nodes_per_year=50, min_font_size=3, edge_threshold=.15):
    global G, orig_G, node_positions, edge_positions
    G, orig_G = load_graph(
        path=f'../../legal-networks-data/{dataset.lower()}/13_cluster_evolution_graph/all_{config_str}.gpickle.gz',
        nodes_per_year=nodes_per_year,
        edge_threshold = edge_threshold,
    )
    
    config = calc_config(G)

    level_node_orders = order_nodes_by_weight(config, G)

    node_positions = calc_node_positions(G, config, level_node_orders)
    edge_positions = calc_edge_positions(G, node_positions, config, level_node_orders)

    alternating_colors(edge_positions, node_positions, G, level_node_orders)
    
#     categeories_df = pd.read_csv(f'../{dataset.upper()}-data/cd_8_cluster_categories/all_{config_str}.csv')
#     categories_colors(edge_positions, node_positions, categeories_df)
    
#     color_merges_splits(edge_positions, node_positions, G)
#     color_births_deaths(node_positions, G)

    color_by_cluster_family(orig_G, node_positions, edge_positions, dataset, edge_threshold)
            
    plt.rcParams['figure.figsize'] = 8, 11
    plt.rcParams['figure.constrained_layout.use'] = True
    fig = plt.figure()
    ax = fig.add_subplot(111)

    for node, position in node_positions.items():
        draw_node(ax=ax, **position)

    edge_positions_list = order_edges_by_weight(edge_positions)
    for edge, position in edge_positions_list:
        draw_edge(ax=ax, **position)

    plt.yticks(config['level_tick_positions'], [l[:4] for l in config['levels']], fontsize=12)
    plt.xticks([])
    filepath_base = f'../data_figures/sankey_{dataset.lower()}_{config_str}'
    if nodes_per_year != 50:
        filepath_base +=f'_miscafter{nodes_per_year or 0}'
    plt.savefig(filepath_base + '.pdf')
    for node, position in node_positions.items():
        community_id = node.split('_')[1]
    #     label = ' '.join(
    #         [x.split('_')[0][:-1] for x in G.nodes[node].get('law_names', '').split(',')[::2]][:3]
    #     )
        draw_label(ax=ax, **position, 
            label=(
                'Sonstige' 
                if node.startswith('misc_') else 
                community_id
            ),
            label_rotation= 0 if node_positions[node]['v_position'] == 'center' else 90,
            min_font_size=min_font_size,
        )
    plt.savefig(filepath_base + '_labels.pdf')
    plt.close()

In [ ]:
config_str = '0-0_1-0_-1_a-infomap_n100_m1-0_s0_c1000'

In [ ]:
plot_evolution_graph('us', config_str, nodes_per_year=50)

In [ ]:
plot_evolution_graph('de', config_str, nodes_per_year=50)

In [ ]:
# for config in ['0-0_1-0_-1_a-infomap_m1-0_s0_c1000'] + [
#     f'0-0_1-0_-1_a-infomap_n{runs}_m1-0_s0_c1000' for runs in list(range(10,150+1,10)) + [200]
# ]:
#     plot_evolution_graph('us', config)
#     print(config, 'done')

In [ ]:
plot_evolution_graph('us', config_str, nodes_per_year=500, min_font_size=None)

In [ ]:
plot_evolution_graph('de', config_str, nodes_per_year=500, min_font_size=None)

In [ ]:
plot_evolution_graph(
    'de', 
    '0-0_1-0_-1_o-2-0_t-paragraph_a-infomap_n100_m1-0_s0_c1000',
    nodes_per_year=50
)

In [ ]:
!gs -dAutoRotatePages=/None \
    -sDEVICE=pdfwrite \
    -dCompatibilityLevel=1.4 \
    -dPDFSETTINGS=/screen \
    -sColorConversionStrategy=Gray \
    -dProcessColorModel=/DeviceGray \
    -dNOPAUSE \
    -dBATCH \
    -sOutputFile=../data_figures/sankey_de_0-0_1-0_-1_o-2-0_t-paragraph_a-infomap_n100_m1-0_s0_c1000_labels_graycolor.pdf \
    ../data_figures/sankey_de_0-0_1-0_-1_o-2-0_t-paragraph_a-infomap_n100_m1-0_s0_c1000_labels.pdf


# Old configs

In [ ]:
# plot_evolution_graph(
#     'de', 
#     '0-0_1-0_-1_o-2-0_t-paragraph_a-louvain_m0-5_s0_c1000',
#     edge_threshold=0.2
# )

In [ ]:
# plot_evolution_graph(
#     'de', 
#     '0-0_1-0_-1_o-1-0_t-paragraph_a-louvain_m1-0_s0_c1000',
#     nodes_per_year=20,
#     edge_threshold=0.05
# )

In [ ]:
# plot_evolution_graph(
#     'de', 
#     '0-0_1-0_-1_a-infomap_n100_m1-0_s0_c1000',
#     nodes_per_year=50
# )

In [ ]:
# plot_evolution_graph(
#     'de', 
#     '0-0_1-0_-1_a-infomap-directed_n100_m1-0_s0_c1000',
#     nodes_per_year=50
# )

In [ ]:
# plot_evolution_graph(
#     'us', 
#     '0-0_1-0_-1_a-infomap_n100_m1-0_s0_c1000',
#     nodes_per_year=50
# )

In [ ]:
# plot_evolution_graph(
#     'us', 
#     '0-0_1-0_-1_a-infomap-directed_n100_m1-0_s0_c1000',
#     nodes_per_year=50
# )

In [ ]:
# plot_evolution_graph(
#     'de', 
#     '0-0_1-0_-1_o-2-0_t-paragraph_a-infomap-directed_n100_m1-0_s0_c1000',
#     nodes_per_year=50
# )